# LUSC morphology + cell boundaries in napari

Raw GEO bundle for the lung squamous cell carcinoma (LUSC) sample (`GSM9509137`) hasn't been through the
clustering/annotation pipeline yet, so unlike `05_napari_morphology.ipynb`/`08_napari_subcluster_epithelial.ipynb` for
`xenium_5k`, there is no `cluster_labels.csv`/`sub_cluster_labels.csv` to color cells by, and no H&E SpatialData store.
This notebook only loads the morphology image pyramid and cell boundary polygons (flat color), with a hover tooltip
showing `cell_id` and a minimap driven off the morphology image instead of H&E.

In [24]:
from pathlib import Path
import os
import time

import numpy as np
import pandas as pd

# Ubuntu + Wayland
os.environ["QT_QPA_PLATFORM"] = "xcb"
os.environ["PYOPENGL_PLATFORM"] = "glx"

import napari
from shapely.geometry import Polygon
from shapely.validation import make_valid

In [32]:
PROJECT = Path.home() / "Projects/xenium_lung"

# GEO bundle ships the morphology/boundaries/cells files gzip-compressed; decompress once into a sibling folder.
BUNDLE_DIR = PROJECT / "data/bundle/xenium_prime_5k_lusc"
RAW_DIR = BUNDLE_DIR / "extracted"

SAMPLE_PREFIX = "GSM9509137_NSCLC_5k_L6_L6"

CELLS_FILE = RAW_DIR / f"{SAMPLE_PREFIX}_cells.parquet"
BOUNDARIES_FILE = RAW_DIR / f"{SAMPLE_PREFIX}_cell_boundaries.parquet"
# _0000 holds channel 0 (DAPI) plus OME-XML references to sibling files _0001.._0003 (one real channel each,
# same convention as xenium_5k's morphology_focus folder) — tifffile's aszarr store stitches them together.
# The OME-XML refers to siblings by their *bare* filename (no SAMPLE_PREFIX), so symlinks with bare names
# are created below for tifffile to resolve them correctly.
MORPHOLOGY_LINKS_DIR = RAW_DIR / "morphology_focus"
MORPHOLOGY_FILE = MORPHOLOGY_LINKS_DIR / "morphology_focus_0000.ome.tif"

print("Exists:", BUNDLE_DIR.exists())
print("Bundle:", BUNDLE_DIR)

Exists: True
Bundle: /home/duydao/Projects/xenium_lung/data/bundle/xenium_prime_5k_lusc


# Decompress the bundle (one-time, idempotent)

In [37]:
import gzip
import shutil

RAW_DIR.mkdir(exist_ok=True)

n_extracted = 0
for gz_file in BUNDLE_DIR.glob("*.gz"):
    target = RAW_DIR / gz_file.stem
    if target.exists():
        continue
    with gzip.open(gz_file, "rb") as src, open(target, "wb") as dst:
        shutil.copyfileobj(src, dst)
    n_extracted += 1

print("Newly extracted:", n_extracted)
print("Cells:", CELLS_FILE, CELLS_FILE.exists())
print("Boundaries:", BOUNDARIES_FILE, BOUNDARIES_FILE.exists())

# tifffile resolves OME sibling files by bare name relative to the *real* directory of the opened file
# (it follows symlinks when computing that directory), so a symlink here doesn't work — copy instead.
MORPHOLOGY_LINKS_DIR.mkdir(exist_ok=True)
for n in range(4):
    bare_name = f"morphology_focus_{n:04d}.ome.tif"
    link = MORPHOLOGY_LINKS_DIR / bare_name
    real_file = RAW_DIR / f"{SAMPLE_PREFIX}_{bare_name}"
    if not link.exists() and real_file.exists():
        os.link(real_file, link)

print("Morphology:", MORPHOLOGY_FILE, MORPHOLOGY_FILE.exists())


Newly extracted: 0
Cells: /home/duydao/Projects/xenium_lung/data/bundle/xenium_prime_5k_lusc/extracted/GSM9509137_NSCLC_5k_L6_L6_cells.parquet True
Boundaries: /home/duydao/Projects/xenium_lung/data/bundle/xenium_prime_5k_lusc/extracted/GSM9509137_NSCLC_5k_L6_L6_cell_boundaries.parquet True
Morphology: /home/duydao/Projects/xenium_lung/data/bundle/xenium_prime_5k_lusc/extracted/morphology_focus/morphology_focus_0000.ome.tif True


# Load the cell table

Raw `cells.parquet` has no `cell_id`/`y`/`x` convention — centroids are `x_centroid`/`y_centroid` in microns.

In [27]:
raw_cells = pd.read_parquet(CELLS_FILE)

cells = raw_cells.rename(columns={"y_centroid": "y", "x_centroid": "x"})[["cell_id", "y", "x"]]

print("Cells:", cells.shape)
print(cells.head())

Cells: (64872, 3)
      cell_id            y           x
0  aaaafkec-1  1842.636108  575.317749
1  aaaamfjm-1  1827.751099  680.041809
2  aaabjfhd-1  1850.443604  578.890564
3  aaachlom-1  1891.183594  780.729614
4  aaadeibp-1  1963.702637  811.958618


# Open napari

In [28]:
viewer = napari.Viewer()

/home/duydao/micromamba/envs/spatial_env/lib/python3.12/site-packages/napari/_qt/qt_event_loop.py:49: UserWarning: System theme detection requires a Qt6 backend. Please switch to PyQt6 or PySide6 to use it.
  theme_type=get_system_theme(),
/home/duydao/micromamba/envs/spatial_env/lib/python3.12/site-packages/napari/_qt/qt_event_loop.py:49: UserWarning: System theme detection requires a Qt6 backend. Please switch to PyQt6 or PySide6 to use it.
  theme_type=get_system_theme(),


# Load morphology image

Same multiscale-pyramid technique as `08_napari_subcluster_epithelial.ipynb`: read the whole OME-TIFF as one
zarr group (keys `"0"`, `"1"`, ... = resolution levels) and wrap each level as a dask array, so only visible
tiles are read from disk.

In [35]:
import re
import tifffile

with tifffile.TiffFile(MORPHOLOGY_FILE) as tif:
    ome_xml = tif.ome_metadata

# cell x/y are in microns; the TIFF pixel grid needs this scale to line up with them
match_x = re.search(r'PhysicalSizeX="([\d.]+)"', ome_xml)
match_y = re.search(r'PhysicalSizeY="([\d.]+)"', ome_xml)

pixel_size_x = float(match_x.group(1)) if match_x else 1.0
pixel_size_y = float(match_y.group(1)) if match_y else 1.0

channel_names = re.findall(r'<Channel[^>]*Name="([^"]+)"', ome_xml)

print("Channel names:", channel_names)
print("Pixel size (x, y):", pixel_size_x, pixel_size_y)

Channel names: ['DAPI', 'ATP1A1/CD45/E-Cadherin', '18S', 'alphaSMA/Vimentin']
Pixel size (x, y): 0.2125 0.2125


In [38]:
import dask.array as da
import zarr

store = tifffile.imread(MORPHOLOGY_FILE, aszarr=True)
group = zarr.open(store, mode="r")
level_keys = sorted(group.array_keys(), key=int)

pyramid = [da.from_array(group[key], chunks="auto") for key in level_keys]

for i, level in enumerate(pyramid):
    print(i, "shape =", level.shape, "dtype =", level.dtype)

0 shape = (4, 20485, 25612) dtype = uint16
1 shape = (4, 10242, 12806) dtype = uint16
2 shape = (4, 5121, 6403) dtype = uint16
3 shape = (4, 2560, 3201) dtype = uint16
4 shape = (4, 1280, 1600) dtype = uint16
5 shape = (4, 640, 800) dtype = uint16
6 shape = (4, 320, 400) dtype = uint16


In [39]:
# A single fixed contrast range doesn't fit every channel's intensity scale. Strided/downsampled sampling can
# miss thin, sparse signal (e.g. membrane outlines), so use true min/max over the full-res level instead —
# dask computes this as a lazy chunked reduction, so it doesn't require loading the whole array into memory.
channel_min = pyramid[0].min(axis=(1, 2)).compute()
channel_max = pyramid[0].max(axis=(1, 2)).compute()
contrast_limits = [
    (float(lo), float(hi) if hi > lo else float(lo) + 1.0)
    for lo, hi in zip(channel_min, channel_max)
]
print("Per-channel contrast limits:", list(zip(channel_names, contrast_limits)))

viewer.add_image(
    pyramid,
    channel_axis=0,
    name=channel_names,
    multiscale=True,
    contrast_limits=contrast_limits,
    scale=(pixel_size_y, pixel_size_x),
    units=("um", "um"),
    visible=False,
)

morph_shape = pyramid[0].shape
print("Morphology world bbox (y, x): [0, 0] to", (morph_shape[-2] * pixel_size_y, morph_shape[-1] * pixel_size_x))

Per-channel contrast limits: [('DAPI', (0.0, 11058.0)), ('ATP1A1/CD45/E-Cadherin', (0.0, 11016.0)), ('18S', (0.0, 7987.0)), ('alphaSMA/Vimentin', (0.0, 11024.0))]
Morphology world bbox (y, x): [0, 0] to (4353.0625, 5442.55)


# Build cell boundary polygons

Unlike `xenium_5k`'s processed `cell_boundaries.parquet` (one WKB geometry per row), the raw bundle's boundary
file is long-form — one row per vertex per `cell_id`. Group by `cell_id` to reassemble each polygon.

In [21]:
raw_boundaries = pd.read_parquet(BOUNDARIES_FILE)

polygon_data = []
polygon_cell_ids = []
n_skipped = 0

for cell_id, group_df in raw_boundaries.groupby("cell_id", sort=False):
    coords = group_df[["vertex_x", "vertex_y"]].to_numpy()

    keep = np.ones(len(coords), dtype=bool)
    keep[1:] = np.any(np.diff(coords, axis=0) != 0, axis=1)
    coords = coords[keep]
    if len(coords) < 3:
        n_skipped += 1
        continue

    # Self-intersecting/degenerate polygons crash vispy's triangulator — repair or drop them.
    geom = Polygon(coords)
    if not geom.is_valid:
        geom = make_valid(geom)
    if geom.geom_type == "MultiPolygon":
        geom = max(geom.geoms, key=lambda g: g.area)
    if geom.geom_type != "Polygon" or geom.is_empty:
        n_skipped += 1
        continue

    # Shapely X,Y -> napari Y,X
    coords_yx = np.asarray(geom.exterior.coords)[:, [1, 0]]

    polygon_data.append(coords_yx)
    polygon_cell_ids.append(cell_id)

print("Polygons:", len(polygon_data))
print("Skipped (invalid/degenerate):", n_skipped)

all_coords = np.concatenate(polygon_data)
print("Boundaries world bbox (y, x):", all_coords.min(axis=0), "to", all_coords.max(axis=0))

Polygons: 64824
Skipped (invalid/degenerate): 48
Boundaries world bbox (y, x): [ 27.83750153 269.23751831] to [4249.14990234 5396.01269531]


# Add cell boundary polygons (flat color — no cluster labels yet)

In [10]:
shapes_layer = viewer.add_shapes(
    polygon_data,
    shape_type="polygon",
    edge_width=0,
    face_color="cyan",
    opacity=0.4,
    units=("um", "um"),
    name=f"Cell boundaries — {len(polygon_data):,}",
    visible=True,
)

KeyboardInterrupt: 

## Hover tooltip showing `cell_id`

In [ ]:
viewer.text_overlay.visible = True
viewer.text_overlay.color = "white"
viewer.text_overlay.font_size = 12

# get_value() on this many polygons is expensive; only run it a few times per second.
TOOLTIP_INTERVAL_S = 0.08
_last_tooltip_time = 0.0


@shapes_layer.mouse_move_callbacks.append
def show_cell_id_tooltip(layer, event):
    global _last_tooltip_time
    now = time.monotonic()
    if now - _last_tooltip_time < TOOLTIP_INTERVAL_S:
        return
    _last_tooltip_time = now

    shape_index = layer.get_value(
        event.position,
        view_direction=event.view_direction,
        dims_displayed=event.dims_displayed,
        world=True,
    )
    if isinstance(shape_index, tuple):
        shape_index = shape_index[0]

    if shape_index is not None:
        viewer.text_overlay.text = f"cell_id: {polygon_cell_ids[shape_index]}"
    else:
        viewer.text_overlay.text = ""

## Overview / minimap panel

No H&E layer here, so the coarsest level of the DAPI channel (channel 0) stands in as the overview thumbnail.

In [ ]:
import matplotlib
matplotlib.use("QtAgg")
from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg
from matplotlib.figure import Figure
from matplotlib.patches import Rectangle
from qtpy.QtCore import QTimer

# Re-running this cell must not stack duplicate dock widgets / camera callbacks.
if "_overview_update_cb" in globals():
    viewer.scene.camera.events.center.disconnect(_overview_update_cb)
    viewer.scene.camera.events.zoom.disconnect(_overview_update_cb)
if "overview_dock_widget" in globals():
    try:
        viewer.window.remove_dock_widget(overview_dock_widget)
    except RuntimeError:
        pass

# Coarsest pyramid level, DAPI channel only, as the overview thumbnail.
morph_thumb = np.asarray(pyramid[-1][0])
morph_thumb_downsample_y = pyramid[0].shape[-2] / pyramid[-1].shape[-2]
morph_thumb_downsample_x = pyramid[0].shape[-1] / pyramid[-1].shape[-1]

world_y_max = morph_thumb.shape[0] * morph_thumb_downsample_y * pixel_size_y
world_x_max = morph_thumb.shape[1] * morph_thumb_downsample_x * pixel_size_x

overview_fig = Figure(figsize=(3, 3))
overview_ax = overview_fig.add_subplot(111)
overview_ax.imshow(
    morph_thumb,
    cmap="gray",
    extent=[0, world_x_max, world_y_max, 0],
)
overview_ax.set_xlim(0, world_x_max)
overview_ax.set_ylim(world_y_max, 0)
overview_ax.set_xticks([])
overview_ax.set_yticks([])
overview_fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

view_rect = Rectangle((0, 0), 1, 1, edgecolor="yellow", facecolor="none", linewidth=1.5)
overview_ax.add_patch(view_rect)

overview_canvas = FigureCanvasQTAgg(overview_fig)
overview_canvas.draw()


def get_canvas_size():
    qt_viewer = getattr(viewer.window, "_qt_viewer", None) or viewer.window.qt_viewer
    size = qt_viewer.canvas.size
    return size[0], size[1]


def _redraw_overview():
    overview_canvas.draw_idle()


# Camera events fire continuously while panning/zooming; debounce with a single-shot Qt timer
# so we redraw at most a few times per second instead of on every event.
_overview_timer = QTimer()
_overview_timer.setSingleShot(True)
_overview_timer.setInterval(30)
_overview_timer.timeout.connect(_redraw_overview)


def _overview_update_cb(event=None):
    camera = viewer.scene.camera
    width_px, height_px = get_canvas_size()
    zoom = camera.zoom
    center_y, center_x = camera.center[-2:]

    half_width = width_px / zoom / 2
    half_height = height_px / zoom / 2

    view_rect.set_xy((center_x - half_width, center_y - half_height))
    view_rect.set_width(2 * half_width)
    view_rect.set_height(2 * half_height)

    _overview_timer.start()


viewer.scene.camera.events.center.connect(_overview_update_cb)
viewer.scene.camera.events.zoom.connect(_overview_update_cb)
_overview_update_cb()
_redraw_overview()

overview_dock_widget = viewer.window.add_dock_widget(overview_canvas, area="left", name="Overview")

In [ ]:
viewer.scale_bar.visible = True